# 03 · Stage 2 — distill into the 6-layer CPU student

The 6-layer student (~100 M parameters, the size of the released Pocket TTS) learns to reproduce the teacher's backbone output **with classifier-free guidance 2.0 baked in**, so inference needs a single 6-layer pass per 80 ms frame — this is what makes it fast on a CPU. Its layers start from the teacher's bottom and top layers; text embedding, voice projection and the sampler head are copied from the teacher.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

REPO = Path("/workspace/EgyPocket-TTS")          # where you cloned the repo
os.environ.setdefault("EGY_DATA", "/workspace/egypocket_data")
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")     # models + HF token persist on the volume
DATA = Path(os.environ["EGY_DATA"])
CATT_DIR = Path("/workspace/catt_tashkeel")      # the uploaded CATT folder
os.chdir(REPO)
sys.path.insert(0, str(REPO))
PY = sys.executable                              # the egypocket conda env
print("python:", PY)
print("repo  :", REPO)
print("data  :", DATA)

In [ ]:
from egypocket import notebook as nb
from egypocket.training.args import load_args
from egypocket.text.frontend import normalize_text

TEACHER_RUN = DATA / "runs/teacher_24l"
ckpts = sorted(TEACHER_RUN.glob("checkpoint_*.pt"))
assert ckpts, "no teacher checkpoint yet"
TEACHER_CKPT = ckpts[-1]          # or pick another one, e.g. TEACHER_RUN / "checkpoint_00024000.pt"
os.environ["TEACHER_CKPT"] = str(TEACHER_CKPT)
assert not nb.is_running(TEACHER_RUN), "stop the teacher first (the GPU is needed)"
print("teacher:", TEACHER_CKPT)

CONFIG = REPO / "configs/distill_6l.yaml"
args = load_args(CONFIG)
RUN = args.run_dir
print(open(CONFIG, encoding="utf-8").read())

In [ ]:
print(nb.start_tensorboard(DATA / "runs"))

In [ ]:
pid = nb.launch_training(CONFIG, RUN, extra_env={"TEACHER_CKPT": str(TEACHER_CKPT)})
print("started, pid", pid)

In [ ]:
print(nb.status(RUN, 30))

In [ ]:
nb.plot_scalars(RUN, ["train/distill_mse", "valid/distill_mse", "optim/lr", "optim/grad_norm", "optim/steps_per_sec", "samples/mean_duration_sec"])

In [ ]:
sentences = [normalize_text(s) for s in args.sample_sentences]
nb.show_samples(RUN, sentences)

## Stop (graceful)

In [ ]:
nb.stop_training(RUN)

In [ ]:
print(nb.status(RUN, 10))

Next: **04_inference.ipynb** exports the student and runs it on the CPU.